In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:
# ------------------------- Basisverzeichnis -------------------------
base_dir = Path.cwd()

# ------------------------- Laden der vorverarbeiteten Daten (4.2) -------------------------
# ../4.2_Preprocessing/Preprocessed_SOM_Ready.csv
input_path_prep = base_dir.parent / "4.2_Preprocessing" / "Preprocessed_SOM_Ready.csv"

if not input_path_prep.exists():
    raise FileNotFoundError(f"Input-File nicht gefunden: {input_path_prep}")

df_full = pd.read_csv(input_path_prep, low_memory=False)

print(f"Daten erfolgreich aus 4.2_Preprocessing geladen! Shape: {df_full.shape}")

Daten erfolgreich aus 4.2_Preprocessing geladen! Shape: (175099, 25)


<div class="alert alert-info">
    <b>4.3 SOM Machine Learning (Imputed Data)</b><br><br>
    <b>Datenquelle:</b><br>
    - 4.2 Preprocessed Data (Imputiert + Log + Gauss)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> (dies sollte schon im Post-Processing von 4.1 erledigt sein, aber wir filtern sicherheitshalber nochmal oder verifizieren es).
    <br>
    <b>Ziel:</b><br>
    - Clustering mittels SOM (Hexagonal).
    - Analyse der Ergebnisse auf Basis der 'geretteten' Daten.
</div>

In [4]:
# ------------------------- Features auswählen (nur Hauptionen) -------------------------
target_ions = ["Na", "Ca", "Mg", "Cl", "HCO3", "SO4"]
features = []
for col in df_full.columns:
    if "_gauss" in col:
        element = col.split("_")[0]
        if element in target_ions:
            features.append(col)

print(f"Input Features für SOM: {features}")

# ------------------------- Filtern -------------------------
df_som = df_full[features + ['temperature_in_c', 'rock_type']].copy()

# ------------------------- Temperatur in numerische Werte umwandeln -------------------------
df_som['temperature_in_c'] = pd.to_numeric(df_som['temperature_in_c'], errors='coerce')

# ------------------------- Filter: Temp >= 10°C oder NaN -------------------------
len_before_temp = len(df_som)
# Logik: (Temp >= 10) ODER (Temp ist NaN)
temp_condition = (df_som['temperature_in_c'] >= 10) | (df_som['temperature_in_c'].isna())
df_som = df_som[temp_condition]

print(f"Filter Temp >= 10°C or NaN: {len_before_temp - len(df_som)} Zeilen entfernt. Verbleibend: {len(df_som)}")

initial_len = len(df_som)
df_som.dropna(subset=features, inplace=True)
print(f"Zeilen mit fehlenden Features entfernt: {initial_len - len(df_som)}. Final: {len(df_som)}")

Input Features für SOM: ['Na_in_mmol/L_gauss', 'Mg_in_mmol/L_gauss', 'Ca_in_mmol/L_gauss', 'Cl_in_mmol/L_gauss', 'SO4_in_mmol/L_gauss', 'HCO3_in_mmol/L_gauss']
Filter Temp >= 10°C or NaN: 10041 Zeilen entfernt. Verbleibend: 165058
Zeilen mit fehlenden Features entfernt: 1. Final: 165057


In [5]:
# ------------------------- MinMax Scaling (SOM-Training) -------------------------
scaler = MinMaxScaler(feature_range=(0, 1))
data_values = df_som[features].values
data_scaled = scaler.fit_transform(data_values)

print("Daten skaliert.")

Daten skaliert.


In [6]:
# ------------------------- SOM-Training -------------------------
som_x = 6 
som_y = 6
input_len = len(features)
sigma = 0.5
learning_rate = 0.5

som = MiniSom(x=som_x, y=som_y, input_len=input_len, 
              sigma=sigma, learning_rate=learning_rate, 
              topology='hexagonal', neighborhood_function='gaussian', 
              activation_distance='euclidean', random_seed=42)

som.random_weights_init(data_scaled)

print(f"Starte SOM Training ({som_x}x{som_y} Hexagonal)...")
som.train_random(data_scaled, 10000) 
print("Training beendet.")

Starte SOM Training (6x6 Hexagonal)...


Training beendet.


In [7]:
# ------------------------- Vorbereitung der Temperaturanalyse -------------------------
winner_coords = []
for x in data_scaled:
    w = som.winner(x)
    winner_coords.append(w)

df_som['som_x'] = [c[0] for c in winner_coords]
df_som['som_y'] = [c[1] for c in winner_coords]

# ------------------------- Berechnung der Mean Temperature Map -------------------------
temp_map = np.full((som_x, som_y), np.nan)

agg = df_som.groupby(['som_x', 'som_y'])['temperature_in_c'].mean()

for (gx, gy), val in agg.items():
    temp_map[gx, gy] = val

print("Mean Temperature Map berechnet.")

Mean Temperature Map berechnet.


In [8]:
# ------------------------- Plot-Funktion für PDF -------------------------
def plot_hex_map_to_axis(ax, data_matrix, title, cmap='viridis', label_suffix='', annotate=False):
    sy, sx = data_matrix.shape
    ax.set_aspect('equal')
    patches = []
    colors = []
    
    for y in range(sy):
        for x in range(sx):
            val = data_matrix[y, x]
            if np.isnan(val): 
                continue 
                
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            
            hex_poly = mpatches.RegularPolygon(
                (center_x, center_y), numVertices=6, radius=radius, 
                orientation=np.radians(30), edgecolor='k', linewidth=0.5
            )
            patches.append(hex_poly)
            colors.append(val)
            
            if annotate:
                ax.text(center_x, center_y, f"{val:.1f}", ha='center', va='center', 
                        fontsize=7, color='black', fontweight='bold')
            
    if not patches:
        return

    collection = PatchCollection(patches, cmap=cmap, alpha=0.9)
    collection.set_array(np.array(colors))
    ax.add_collection(collection)
    
    ax.set_xlim(-0.5, sx + 0.5)
    ax.set_ylim(-0.5, sy * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return collection

In [9]:
# ------------------------- PDF-Berichterstellung -------------------------
# Output in lokalen Ordner
output_root = base_dir / "MiniSom_Results"
output_root.mkdir(exist_ok=True)

# ------------------------- Erstellung des Run-Ordners -------------------------
current_ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_folder = output_root / f"Run_{current_ts}"
run_folder.mkdir(parents=True, exist_ok=True)
print(f"Output Folder: {run_folder}")


pdf_path = run_folder / "SOM_Report_Temperature_Imputed.pdf"
print(f"Erstelle Report: {pdf_path} ...")

# ------------------------- Anzahl berechnen -------------------------
temp_count = df_som['temperature_in_c'].count()
with PdfPages(pdf_path) as pdf:
    
    # ------------------------- Beschreibungsseite -------------------------
    plt.figure(figsize=(8, 6))
    plt.text(0.5, 0.5, f"SOM Analysis (Imputed Data)\n\nHexagonal Self-Organizing Map (6x6)\nInput: Major Ions (Log+Gauss)\nOverlay: Temperature (Mean & Dist)\nFILTER: Only Temp > 10\u00b0C\n\nTotal Data Points: {temp_count}", 
             ha='center', va='center', fontsize=20)
    plt.axis('off')
    pdf.savefig()
    plt.close()
    
    # ------------------------- Abstandsmatrix -------------------------
    fig, ax = plt.subplots(figsize=(8, 8))
    u_matrix = som.distance_map()
    col_u = plot_hex_map_to_axis(ax, u_matrix, "U-Matrix (Distance Map)", cmap='bone_r')
    plt.colorbar(col_u, ax=ax, fraction=0.046, pad=0.04, label='Distance')
    pdf.savefig(fig)
    plt.close(fig)
    
    # ------------------------- Durchschnittstemperatur -------------------------
    fig, ax = plt.subplots(figsize=(10, 10))
    col_t = plot_hex_map_to_axis(ax, temp_map, "Mean Temperature per Cluster (> 10°C)", cmap='coolwarm', annotate=True)
    plt.colorbar(col_t, ax=ax, fraction=0.046, pad=0.04, label='Temp °C')
    pdf.savefig(fig)
    plt.close(fig)
    
    # ------------------------- Component Plane -------------------------
    W = som.get_weights()
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, feature in enumerate(features):
        short = feature.split("_")[0]
        col_c = plot_hex_map_to_axis(axes[i], W[:,:,i], f"{short}", cmap='viridis')
    plt.suptitle("Component Planes (Ion Distribution)")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print("Report fertiggestellt.")

Output Folder: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Machine-Learning\MiniSom_Results\Run_2026-01-07_00-39-51
Erstelle Report: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Machine-Learning\MiniSom_Results\Run_2026-01-07_00-39-51\SOM_Report_Temperature_Imputed.pdf ...


Report fertiggestellt.
